# Baseline: популярные книги среди кандидатов

Этот ноутбук проверяет полный маршрут: загрузка данных, расчёт популярности, формирование топ-5, локальная метрика и создание файла для test.

## 1. Загружаем библиотеки и файлы

In [ ]:
import pandas as pd

from metric import ndcg_at_5

In [ ]:
users = pd.read_csv("users.csv")
books = pd.read_csv("books.csv")
interactions = pd.read_json("interactions.jsonl", lines=True)
candidates = pd.read_csv("candidates.csv")

print("Читателей:", len(users))
print("Книг:", len(books))
print("Взаимодействий:", len(interactions))
print("Пар-кандидатов:", len(candidates))

## 2. Считаем популярность книг

Популярность книги — количество исторических взаимодействий с ней. Baseline не учитывает личные интересы читателя, но даёт первую точку отсчёта.

In [ ]:
book_popularity = interactions["book_id"].value_counts()
book_popularity.head()

## 3. Создаём функцию для топ-5

Для каждого читателя берём только его книги-кандидаты, добавляем популярность, сортируем и выбираем первые пять.

In [ ]:
def make_popularity_submission(candidates_part, popularity):
    work = candidates_part.copy()
    work["popularity"] = work["book_id"].map(popularity).fillna(0)
    work = work.sort_values(
        ["reader_id", "popularity", "book_id"],
        ascending=[True, False, True]
    )
    top_5 = work.groupby("reader_id").head(5)
    submission = (
        top_5.groupby("reader_id")["book_id"]
        .apply(lambda values: " ".join(values))
        .reset_index(name="recommendations")
    )
    return submission

## 4. Проверяем baseline на valid

In [ ]:
valid_candidates = candidates[candidates["split"] == "valid"]
valid_submission = make_popularity_submission(valid_candidates, book_popularity)
valid_submission.to_csv("baseline_valid_submission.csv", index=False)
valid_submission.head()

In [ ]:
valid_score = ndcg_at_5(
    "valid_solution.csv",
    "baseline_valid_submission.csv",
    "candidates.csv"
)
print("NDCG@5 baseline:", round(valid_score, 6))

## 5. Создаём файл для test

Правильных ответов для test в комплекте участника нет. Файл можно проверить только по формату, а итоговую оценку выдаёт платформа.

In [ ]:
test_candidates = candidates[candidates["split"] == "test"]
test_submission = make_popularity_submission(test_candidates, book_popularity)
test_submission.to_csv("submission.csv", index=False)
test_submission.head()

Baseline готов. Его ограничение: рекомендации строятся только по общей популярности, поэтому у разных читателей списки часто похожи.